In [2]:
Install dependencies

SyntaxError: invalid syntax (3470324418.py, line 1)

In [3]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q langchain-core
!pip install -q sentence_transformers
!pip install -q langchain-chroma
!pip install -q transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [4]:
import langchain
import transformers
import sentence_transformers

print("OK")

OK


k


In [5]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-06-03 08:51:33--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json’

ori_pqal.json       100%[===================>]   2.46M  --.-KB/s    in 0.06s   

2026-06-03 08:51:34 (38.1 MB/s) - ‘ori_pqal.json’ saved [2584787/2584787]



In [6]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T

tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(
        lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]),
        axis=1
    ),
    "year": tmp_data.YEAR
})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index
})

print("Questions:", len(questions))
print("Documents:", len(documents))

Questions: 890
Documents: 890


In [7]:
print("\nSample Question:")
print(questions.iloc[0].question)

print("\nSample Document:")
print(documents.iloc[0].abstract[:500])


Sample Question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Sample Document:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


Step 2: Configure your LangChain LM
⚙  Task 2.1. Select a language model
Select a language model that will act as the generative model in your RAG pipeline. You can browse for different HuggingFace models on their webpage.

Some interesting models (e.g. Llama 3.2) may require that you apply for access. This process is usually quite fast, while it may require that you create an account on Hugging Face (it is free). To use a gated model you need to generate a personal HF token and put it as a secret in your notebook (if using Colab). Make sure that the token has enabled “Read access to contents of all public gated repos you can access”.

You can load the HuggingFace language model using HuggingFacePipeline.from_model_id

When calling HuggingFacePipeline, set return_full_text=False to only return the assistant’s response, and call model.invoke(your_prompt) to retrieve the text of the output.

Sanity check: Prompt your LangChain model and confirm that it returns a reasonable output.

In [8]:
from langchain_huggingface import HuggingFacePipeline

model = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 50,
        "return_full_text": False
    }
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
response = model.invoke(
    "Answer briefly: What is machine learning?"
)

print(response)

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Part 3: Set up the document database
🎓  Task 3.1. Embedding model
First, you need a model to embed the documents in the retrieval corpus. Here, we recommend using the HuggingFaceEmbeddings function.

Sanity check: Pass a text passage to the embedding model by calling embed_query and evaluate its shape. It should be of the shape (embedding_dim,).



In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
embedding = embeddings.embed_query(
    "What is programmed cell death?"
)

print(type(embedding))
print(len(embedding))

<class 'list'>
384


⚙  Task 3.2. Chunking
Second, you need to chunk the documents in your retrieval corpus, as some likely are too long for the embedding model. Here, you can use the RecursiveCharacterTextSplitter as a start. The retrieval corpus is given by documents.abstract, so you can use create_documents on the text splitter with the retrieval corpus to create LangChain Document objects, and then use split_documents to create text chunks that will be used in creating the vector store.

For evaluation in Step 5, we recommend saving the document id as metadatas when creating the document:

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadata=metadatas)

Sanity check: Print some samples from the text chunks and check that it makes sense. This way, you might be able to get a feeling for a good chunk size.

Reflection: How do you think design choices related to chunking can affect the quality of RAG systems?

In [10]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [12]:
!pip install -q langchain-text-splitters

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

metadatas = [{"id": idx} for idx in documents.index]

texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas
)

chunks = text_splitter.split_documents(texts)

In [15]:
print("Number of chunks:", len(chunks))

print("\nMetadata:")
print(chunks[0].metadata)

print("\nFirst chunk:")
print(chunks[0].page_content[:300])

Number of chunks: 3510

Metadata:
{'id': 21645374}

First chunk:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel


🎓  Task 3.3. Define a vector store
Third, you need a vector store to store the documents and corresponding embeddings. There are many document databases and retrievers to play around with. As a start, you can use the Chroma vector store with cosine similarity as the distance metric.

When building your vector store, pass the embedding model in Step 3.1 as the embedding model and use the text chunks in Step 3.2 as the documents in the vector store. To add documents in the vector store, you can Use Chroma.from_documents when creating the vector store or use vector_store.add_documents after creating the vector store.

Sanity check: Query your vector store as follows and check that the results make sense:

In [16]:
from langchain_chroma import Chroma

In [17]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store created!")

Vector store created!


In [18]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?",
    k=3
)

for i, (res, score) in enumerate(results):
    print("=" * 80)
    print("Result", i + 1)
    print("Score:", score)
    print("Metadata:", res.metadata)
    print(res.page_content[:300])

Result 1
Score: 0.7619991898536682
Metadata: {'id': 21645374}
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel
Result 2
Score: 1.2089937925338745
Metadata: {'id': 15223779}
mutations in exon 11 of the KIT gene were found. On the contrary, expression of the stem cell growth factor (c-kit ligand) was detected in all three uveal melanoma cell lines, suggesting the presence of autocrine (paracrine) stimulation pathways. Treatment of uveal melanoma cell lines with STI571, w
Result 3
Score: 1.270958423614502
Metadata: {'id': 18158048}
syndrome can harbor viable germ cell elements or seminiferous tubules. The exact fate of these residual elements remains unknown; however, there may exist the potential for malignant transformation. Given the pote

Part 4: Implementing the system
🎓  Task 4.1. Defining the full RAG pipeline
In this and the following steps, we will gradually build a RAG chain.

There could be two options of building a RAG chain, and you can choose either one of them to build your own RAG:

Option A: Build a RAG agent based on the official LangChain guide: here. Here we will use a two-step chain, in which we will run a search in the vector store, and incorporate the result as context for LLM queries.

Option B: Build a RAG chain using LangChain Expression Language (LCEL) based on a LangChain Open Tutorial: here. Here we will use the RunnableParallel class to build a RAG chain that will also return the retrieved document.



In [19]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)

In [20]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Answer the following medical question using the provided context.

Only answer with YES or NO.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [22]:
rag_chain = RunnableParallel(
    {
        "context": retriever,
        "question": lambda x: x
    }
).assign(
    answer=(
        prompt
        | model
        | StrOutputParser()
    )
)

In [23]:
query = questions.iloc[0].question

result = rag_chain.invoke(query)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED DOC:")
print(result["context"][0].page_content[:500])

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

ANSWER:

Option:
YES

Explanation:
The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of this plant consist of a latticework of longitud

RETRIEVED DOC:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has


Part 5: Evaluate RAG on the dataset
We conclude the assignment by evaluating the RAG agent with the given dataset.

🎓  Task 5.1. High-level evaluation
Evaluate your full RAG pipeline on the medical questions (questions.question) and corresponding gold labels (questions.gold_label).

Since the gold labels can be casted to a binary variable (yes/no) you may use the f1 and/or accuracy metrics.

We expect the model to give answers of “Yes” or “No”, but it can happen that the model gives random answers. In this case, one way to perform the evaluation is to keep track of the number of valid answers and do evaluation only on the valid answers.

As a baseline, run the same LM without context and compare the performance of the two setups. You can use the same evaluation method as the previous RAG evaluation. Did the retrieval help?

In [24]:
import re
from sklearn.metrics import accuracy_score

predictions = []
golds = []

sample_size = 100

for i in range(sample_size):

    question = questions.iloc[i].question
    gold = questions.iloc[i].gold_label

    try:
        result = rag_chain.invoke(question)

        answer = result["answer"].upper()

        if "YES" in answer:
            pred = "yes"
        elif "NO" in answer:
            pred = "no"
        else:
            continue

        predictions.append(pred)
        golds.append(gold)

    except Exception:
        continue

print("Valid predictions:", len(predictions))
print("Accuracy:", accuracy_score(golds, predictions))

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Valid predictions: 94
Accuracy: 0.6276595744680851


In [ ]:
from sklearn.metrics import accuracy_score

baseline_preds = []
baseline_golds = []

for i in range(100):

    question = questions.iloc[i].question
    gold = questions.iloc[i].gold_label

    prompt = f"""
Answer YES or NO only.

Question:
{question}
"""

    try:

        answer = model.invoke(prompt).upper()

        if "YES" in answer:
            pred = "yes"
        elif "NO" in answer:
            pred = "no"
        else:
            continue

        baseline_preds.append(pred)
        baseline_golds.append(gold)

    except:
        continue

print("Baseline Accuracy:",
      accuracy_score(baseline_golds,
                     baseline_preds))

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
correct = 0

for i in range(100):

    q = questions.iloc[i]

    retrieved = retriever.invoke(
        q.question
    )[0].metadata["id"]

    if retrieved == q.gold_document_id:
        correct += 1

print(
    "Retrieval Accuracy:",
    correct / 100
)

In [ ]:
for i in range(3):

    q = questions.iloc[i]

    result = rag_chain.invoke(
        q.question
    )

    print("="*80)

    print("Question:")
    print(q.question)

    print("\nGold:")
    print(q.gold_label)

    print("\nPrediction:")
    print(result["answer"])

    print("\nRetrieved ID:")
    print(result["context"][0].metadata)

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Gold:
yes

Prediction:

Answer:
No, mitochondria do not play a role in remodelling lace plant leaves during programmed cell death.

Retrieved ID:
{'id': 21645374}
